# Geo region added to load_qog_timeseries()
## COMPLETE

In [1]:
using Revise
using InteractiveUtils

includet("phase0/functions/load_phase0.jl")


In [2]:
function load_dataframes()
    df = load_qog_timeseries()
    meta_df = CSV.read(PATH_METADATA_JOINED, DataFrame)
    return df, meta_df
end

load_dataframes (generic function with 1 method)

In [3]:
df, meta_df = load_dataframes();

QoG Time-Series Loader Pipeline
Input: data/qog_std_ts_jan25.arrow

>>> Step 1/5: Loading raw data with identity promotion...
    Loaded: 12391 rows × 2010 columns
QoG Time-Series Loader Pipeline
Input: data/qog_std_ts_jan25.arrow

>>> Step 1/5: Loading raw data with identity promotion...
    Loaded: 12391 rows × 2010 columns
    ✓ ggis_rowid assigned

>>> Step 2/5: Previewing rescue collisions...
>>> RESCUE COLLISION PREVIEW (informational — NO rows will be deleted):
    Total (ccode, year) pairs with >1 row: 22

    By entity combination:
      VDR + VNM: 22 years (1955-1976)
        Years: 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976

    NOTE: Use `ggis_rowid` as unique key, or (ident_ccode, ident_year, ident_ccodealp)
    ⚠️  Found 1 collision group(s)
    (See year details above)

>>> Step 3/5: Rescuing historical ccodes...
>>> Historical Ccode Rescue:
    Missing before: 234
    Rescued: 234
  

In [4]:
geo_df = CSV.read(PATH_GEO_LOOKUP, DataFrame);

In [5]:
 dataframe_summaries()

=== DataFrame: REGION_LABELS ===
10×2 DataFrame

=== DataFrame: df ===
12391×2014 DataFrame

=== DataFrame: geo_df ===
209×10 DataFrame

=== DataFrame: meta_df ===
2010×8 DataFrame



In [6]:
function diagnose_ghost_entities(df_rescued::DataFrame, lookup_path::String)
    # Load the geographic lookup
    df_lookup = CSV.read(lookup_path, DataFrame)
    
    # Isolate unique identity coordinates from the data
    df_spine = unique(df_rescued[:, [:ident_ccode, :ident_cname]])
    
    # Find records in data that are NOT in the lookup
    df_missing = antijoin(df_spine, df_lookup, on = :ident_ccode)
    
    if nrow(df_missing) > 0
        println("⚠️ Found $(nrow(df_missing)) Ghost Entities requiring manual mapping:")
        display(df_missing)
    else
        println("✅ All entities successfully mapped to UN sub-regions.")
    end
    
    return df_missing
end

diagnose_ghost_entities (generic function with 1 method)

In [7]:

ghost_entities = diagnose_ghost_entities(df, PATH_GEO_LOOKUP);

⚠️ Found 1 Ghost Entities requiring manual mapping:


Row,ident_ccode,ident_cname
,Int64?,String
1,9156,Tibet


In [24]:
function apply_georegion_layer(df::DataFrame, geo_df::DataFrame)
    df_enriched = copy(df)
    
    # 1. Standard Join
    map_ref = select(geo_df, [:ident_ccode, :un_subregion_code])
    df_enriched = leftjoin(df_enriched, map_ref, on = :ident_ccode)
    
    # 2. Mapping Function with corrected key
    function resolve_code(ccode, un_code)
        if !ismissing(ccode) && haskey(GHOST_REGION_MAP, ccode)
            return Int64(GHOST_REGION_MAP[ccode].code)
        end
        return un_code
    end
    
    df_enriched[!, :ggis_un_subregion_code] = map(
        (c, u) -> resolve_code(c, u), 
        df_enriched.ident_ccode, 
        df_enriched.un_subregion_code
    )
    
    # 3. Final Verification
    missing_indices = findall(ismissing, df_enriched.ggis_un_subregion_code)
    if !isempty(missing_indices)
        bad_entities = unique(df_enriched[missing_indices, [:ident_ccode, :ident_cname]])
        @error "Phase 0 Failure: Spine not saturated." Unmapped=bad_entities
        error("Process Halted.")
    end
    
    select!(df_enriched, Not(:un_subregion_code))
    println("✅ Phase 0: 100% Saturation. All rows mapped to UN Sub-regions.")
    return df_enriched
end

# df = apply_georegion_layer_final(df, geo_df)

apply_georegion_layer (generic function with 1 method)

In [25]:
df_test = apply_georegion_layer(df, geo_df);

✅ Phase 0: 100% Saturation. All rows mapped to UN Sub-regions.


In [9]:
cols_in_df = propertynames(df)
println("Verification - Is :ggis_un_subregion_code present? ", :ggis_un_subregion_code in cols_in_df)

Verification - Is :ggis_un_subregion_code present? true


In [27]:
# 1. Selection of Columns for verification
verify_cols = [:ident_ccode, :ident_ccodealp, :ident_ccodealp_year, :ggis_region, :ggis_un_subregion_code]

# 2. Extract Tibet rows
df_tibet = filter(row -> row.ident_cname == "Tibet", df_test)

# 3. Take a random sample of 20 other rows
# We use replace=false to ensure unique observations
sample_indices = sample(1:nrow(df_test), 20, replace=false)
df_sample = df_test[sample_indices, :]

# 4. Combine and display
verification_view = vcat(df_sample, df_tibet)
select(verification_view, verify_cols)

Row,ident_ccode,ident_ccodealp,ident_ccodealp_year,ggis_region,ggis_un_subregion_code
,Int64?,String,String,Int64?,Int64
1,887,YEM,YEM02,3,145
2,100,BGR,BGR80,1,151
3,780,TTO,TTO14,10,419
4,524,NPL,NPL74,8,34
5,887,YEM,YEM74,3,145
6,144,LKA,LKA15,8,34
7,492,MCO,MCO59,5,155
8,246,FIN,FIN18,5,154
9,818,EGY,EGY46,3,15
